In [ ]:
import numpy as np
import tqdm
import ipywidgets
import matplotlib.pyplot as plt
from scipy.constants import k, e
from scipy.interpolate import RegularGridInterpolator
import sys
sys.path.append('../../sedov_theory/python/')
from sedov_theory import SedovTalorProblem
from analysis_tool import CastroSimulation

In [ ]:
run_dir = '../run/512'
file_start = 'plt_1d_'

In [ ]:
cs = CastroSimulation(run_dir, file_start)

# Compare data with theory

In [ ]:
# Calculate analytical solution
gamma = 5./3
rho = 1.67e-6 # g / cm^3
E = 4400 # erg / cm

sol = SedovTalorProblem(gamma, E, rho)

In [ ]:
plt.figure(figsize=(5, 4))
plt.clf()
level = 0
time = 2e-9

labels = {
    'rho_Hn': 'Density of Hydrogen atoms',
    'rho_Hp': 'Hydrogen ions',
    'density': 'Total Hydrogen density',
}
colors = {
    'rho_Hn': 'gray',
    'rho_Hp': 'blue',
    'density': 'black'
}

for quantity in ['density', 'rho_Hp']:
    r, q, t = cs.extract_data( time, quantity, level)
    # Convert r from cm to microns
    # Convert massic density in cc to number density in cc
    plt.plot(1e4*r, q/1.67e-6, '-o', label=labels[quantity], color=colors[quantity])

# Plot Sedov-Taylor solution
r_th = r
plt.plot( 1e4*r_th, sol.evaluate( 'density', r, t )/1.67e-6, 'k--', label='Sedov-Taylor theory' )
plt.legend(loc=2)

plt.ylabel('Density (10$^{18}$ cm$^{-3}$)')
plt.xlabel('r ($\mu m$)')
plt.title(f'Density at t={t*1.e9:.1f} ns')
plt.xlim(0, 150)

In [ ]:
# plot data at difference resolutions
quantity = 'temperature'
t = 0.e-9

plt.clf()
for level in range(3,-1,-1):
    r, q, t = cs.extract_data( t, quantity, level)
    plt.plot(r, q, '-o', label='level=%d'%level)
    if level == 3:
        r_th = r
        t_th = t
#plt.plot( r_th, sol.evaluate( quantity, r_th, t_th ), 'k--', label='analytical' )
plt.legend(loc=0)

In [ ]:
t = 0e-9

r, T, t = cs.extract_data( t, 'Temp', level=0)
plt.plot( r*1e4, k*T/e, label='Temperature' )
plt.legend(loc=3)
plt.ylim(0,30)
plt.ylabel('Temperature (eV)')
plt.grid()
plt.twinx()
r, n, t = cs.extract_data( t, 'density', level=0)
plt.plot( r*1.e4, n/1.67e-6 * 1.e24, color='r', label='heavies density' )
plt.xlim(0,100)
r, n, t = cs.extract_data( t, 'rho_Hp', level=0)
plt.plot( r*1.e4, n/1.67e-6 * 1.e24, color='g' , label='electrons density' )
plt.legend(loc=5)
plt.xlim(0,100)
plt.ylim(0, 1.1e24)
plt.ylabel('Density (m^-3)')
plt.xlabel('r (microns)')

In [ ]:
t = 1.2e-9

r, T, t = cs.extract_data( t, 'Temp', level=3)
plt.plot( r*1e-2, k*T/e )
plt.ylim(0,30)
plt.grid()
plt.twinx()
r, n, t = cs.extract_data( t, 'density', level=3)
plt.plot( r*1.e-2, n, color='r' )
plt.xlim(0,100e-6)

In [ ]:
# Extract data from different time
# Note that time is not regularly spaced
q_arr = []
rmax_arr = []
for time in tqdm.tqdm( cs.output_times ):
    r, q, t = cs.extract_data(time, 'rho_Hp', level=0)
    rmax = r[np.argmax(q)]
    rmax_arr.append(rmax)
    q_arr.append(q)
q_arr = np.stack(q_arr)
t_arr = cs.output_times.copy()
r_arr = r

# Interpolate on a grid with regularly-spaced time
interp = RegularGridInterpolator(points=(t_arr, r_arr), values=q_arr, bounds_error=False, fill_value=None)
t_interp, r_interp = np.meshgrid(
    np.linspace(0, t_arr.max(), 1001),
    np.linspace(0, r_arr.max(), 1001), indexing='ij')
q_interp = interp((t_interp, r_interp))

In [ ]:
plt.figure(figsize=(6, 2))
plt.imshow(q_interp.T/1.67e-6, origin='lower',
           extent=[0, t_arr.max()*1.e9, 0, r_arr.max()*1.e4],
           aspect='auto', vmax=5)
cb = plt.colorbar()
cb.set_label('H density (10$^{18}$ cm$^{-3}$)')
#plt.plot(t_arr, rmax_arr, 'r-')
#r_analytical = sol.blast_radius(t_arr)
#plt.plot(t_arr*1.e9, r_analytical*1.e4, 'r--', label='Sedov-Taylor theory')
#plt.legend(loc=0)
plt.ylabel('r ($\mu m$)')
plt.xlabel('t (ns)')
plt.ylim(0,300)

In [ ]:
np.save('processesOFF_electrons_512.npy', q_interp.T/1.67e-6 * 1.e24)

In [ ]:
n_512 = np.load('processesOFF_heavies_512.npy')
n_1024 = np.load('processesOFF_heavies_1024.npy')
n_2048 = np.load('processesOFF_heavies_2048.npy')
n_d = np.load('1T_AllOff_heavies.npy')

In [ ]:
n_512 = np.load('processesOFF_electrons_512.npy')
n_1024 = np.load('processesOFF_electrons_1024.npy')
n_2048 = np.load('processesOFF_electrons_2048.npy')
n_d = np.load('1T_AllOff_electrons.npy')

In [ ]:
def plot(it=500):
    r = np.linspace(0, r_arr.max()*1.e4, 1001)
    plt.plot( r, n_512[:,it], label='Castro, 512 grid pts')
    plt.plot( r, n_1024[:,it], label='Castro, 1024 grid pts')
    plt.plot( r, n_2048[:,it], label='Castro, 2048 grid pts')
    plt.plot( r, n_d[:,it], label='Comsol' )
    #plt.ylim( 0, 2e24 )
    plt.xlim(0,100)
    plt.legend(loc=0)
    plt.ylabel('Density (m^-3)')
    plt.xlabel('r (microns)')

In [ ]:
plot(100)
plt.title('t = 1 ns')
plt.grid()